## Imports & config

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- paths ---
DATA_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Raw"
PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
os.makedirs(PROC_DIR, exist_ok=True)

# --- shared spine: same target & split as the whole team ---
import sys
sys.path.append("/content/drive/MyDrive/Machine_learning_project/src")
import common

target = common.load_target()               # id, price, log_price
train_ids, test_ids = common.load_split()   # the fixed split

## Load data

In [6]:
listings = pd.read_csv(f"{DATA_DIR}/listings.csv")
reviews  = pd.read_csv(f"{DATA_DIR}/reviews.csv")

print("listings:", listings.shape, "| reviews:", reviews.shape)

listings: (6996, 79) | reviews: (575824, 6)


In [28]:
reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,6606,5664,2009-07-17,18085,Vivian,"The Urban Cottage is comfortable, beautiful, f..."
1,6606,338761,2011-06-27,434031,Elliott,Joyce was a wonderful host and the urban cotta...
2,6606,467904,2011-08-22,976182,Allegra,Beautiful cottage and warm hospitality from Jo...
3,6606,480017,2011-08-27,997921,Brittney,"Joyce is a wonderful host! She is warm, helpfu..."
4,6606,487278,2011-08-30,206901,Pascal,Joyce's cottage is the perfect Seattle locatio...


## Preprocess: build one text row per listing

In [7]:
# (a) aggregate reviews -> one row per listing
# removes reviews whose comment is missing.
rev = reviews.dropna(subset=["comments"]).copy()
# Convert everything to strings
rev["comments"] = rev["comments"].astype(str)
# Group by listing
agg = (
    rev.groupby("listing_id")
    # Aggregate all reviews
       .agg(reviews_concat=("comments", lambda s: " ".join(s)),
            # Counts how many reviews exist.
            review_count=("comments", "size"))
       .reset_index()
       .rename(columns={"listing_id": "id"})
)

# (b) start from the usable listings (target defines who's in), attach text
text_df = (
    target[["id"]]
      .merge(listings[["id", "name", "description"]], on="id", how="left")
      .merge(agg, on="id", how="left")
)

# (c) fill gaps (listings with no reviews / missing text)
text_df["reviews_concat"] = text_df["reviews_concat"].fillna("")
text_df["review_count"]   = text_df["review_count"].fillna(0).astype(int)
text_df["has_reviews"]    = (text_df["review_count"] > 0).astype(int)
text_df["name"]           = text_df["name"].fillna("")
text_df["description"]    = text_df["description"].fillna("")

# (d) one combined text field per listing
text_df["full_text"] = (
    text_df["name"] + " " + text_df["description"] + " " + text_df["reviews_concat"]
).str.strip()

text_df = text_df.set_index("id")
print("text table:", text_df.shape)

text table: (6158, 6)


In [8]:
text_df.head()

,name,description,reviews_concat,review_count,has_reviews,full_text
id,,,,,,
6606,"Fab, private seattle urban cottage!","This tiny cottage is only 15x10, but it has ev...","The Urban Cottage is comfortable, beautiful, f...",161,1,"Fab, private seattle urban cottage! This tiny ..."
9419,Glorious sun room w/ memory foambed,This beautiful double room features sun filled...,"If you love art, animals, and yoga, this is th...",220,1,Glorious sun room w/ memory foambed This beaut...
11012,"the orange house, quiet 'n central",,I really enjoyed my stay in Joyce's home! It's...,98,1,"the orange house, quiet 'n central I really e..."
25002,Beautiful Private Spot in North Ballard,"-Great eating , Delancey, Fat Hen, 3 blocks aw...","First, would I stay here again - YES. The roo...",1139,1,Beautiful Private Spot in North Ballard -Grea...
26795,Lake Union Cottage - Shore and City View,"This sunny, corner lot is directly across from...",We had a wonderful time the it was not a hotel...,64,1,Lake Union Cottage - Shore and City View This ...


## Feature extraction

In [9]:
all_ids    = text_df.index.values
train_mask = np.isin(all_ids, train_ids)

# (a) TF-IDF — fit on train text only
tfidf = TfidfVectorizer(
    max_features=5000, min_df=5, max_df=0.8,
    stop_words="english", ngram_range=(1, 2),
)
tfidf.fit(text_df.loc[train_ids, "full_text"])          # FIT: train only
X_tfidf = tfidf.transform(text_df["full_text"])         # APPLY: all rows -> (6158, 5000)

# (b) hand-built features: scale review_count (fit on train), keep has_reviews as 0/1
scaler = StandardScaler()
rc = text_df["review_count"].values.reshape(-1, 1).astype(float)
scaler.fit(rc[train_mask])                              # FIT: train only
rc_scaled = scaler.transform(rc)
hr = text_df["has_reviews"].values.reshape(-1, 1).astype(float)

# (c) combine -> final feature matrix
X_text = sparse.hstack([X_tfidf, sparse.csr_matrix(np.hstack([rc_scaled, hr]))]).tocsr()
print("feature matrix:", X_text.shape)                 # (6158, 5002)

feature matrix: (6158, 5002)


## Helper + modeling

In [10]:
# helper: map listing ids -> matrix row positions (aligns features to labels)
row_of = {id_: i for i, id_ in enumerate(all_ids)}
def rows(ids): return [row_of[i] for i in ids]

y = target.set_index("id")["log_price"]
y_train, y_test = y.loc[train_ids].values, y.loc[test_ids].values
X_train, X_test = X_text[rows(train_ids)], X_text[rows(test_ids)]

# train + predict
model = Ridge(alpha=1.0).fit(X_train, y_train)
pred_log = model.predict(X_test)

# evaluate — log scale (primary) + dollars (interpretable)
r2       = r2_score(y_test, pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, pred_log))
rmse_usd = np.sqrt(mean_squared_error(np.exp(y_test), np.exp(pred_log)))
mae_usd  = mean_absolute_error(np.exp(y_test), np.exp(pred_log))
print(f"R²: {r2:.3f} | log-RMSE: {rmse_log:.3f} | RMSE ${rmse_usd:.2f} | MAE ${mae_usd:.2f}")

R²: 0.586 | log-RMSE: 0.371 | RMSE $108.95 | MAE $52.95


## Save features & model

In [ ]:
# ============================================================
# 5. SAVE — feature matrix (for fusion) + fitted transforms
# ============================================================
import joblib

# feature matrix + its id order (so anyone can realign to the split)
sparse.save_npz(f"{PROC_DIR}/text_features_v1.npz", X_text)
np.save(f"{PROC_DIR}/text_features_v1_ids.npy", all_ids)

# the fitted transforms + model (so results are reproducible without refitting)
joblib.dump(tfidf,  f"{PROC_DIR}/text_tfidf.joblib")
joblib.dump(scaler, f"{PROC_DIR}/text_scaler.joblib")
joblib.dump(model,  f"{PROC_DIR}/text_model_ridge.joblib")

with open(f"{PROC_DIR}/text_tfidf_vocab.json", "w") as f:
    json.dump(tfidf.get_feature_names_out().tolist(), f)

print("saved features, transforms, and model to", PROC_DIR)

In [34]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

xgb = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb.fit(X_train, y_train)
pred = xgb.predict(X_test)

r2   = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"XGBoost (TF-IDF)     R²: {r2:.3f} | log-RMSE: {rmse:.3f}")

XGBoost (TF-IDF)     R²: 0.613 | log-RMSE: 0.359


## Trash

In [4]:
import sys
sys.path.append("/content/drive/MyDrive/Machine_learning_project/src")   # where common.py lives

import common

target = common.load_target()               # id, price, log_price
train_ids, test_ids = common.load_split()   # the fixed split
print(target.shape, len(train_ids), len(test_ids))

(6158, 3) 4926 1232


In [5]:
import pandas as pd

listings = pd.read_csv(f"{DATA_DIR}/listings.csv")
reviews  = pd.read_csv(f"{DATA_DIR}/reviews.csv")

# --- inspect reviews (the file with the aggregation trap) ---

# 1. shape + columns
print("reviews shape:", reviews.shape)
print("columns:", reviews.columns.tolist())

# 2. how many unique listings have at least one review
print(reviews["listing_id"].nunique(), "listings have reviews")

# 3. reviews-per-listing distribution
print(reviews["listing_id"].value_counts().describe())

# 4. peek at real review text
print(reviews["comments"].dropna().head(3).tolist())

reviews shape: (575824, 6)
columns: ['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']
6109 listings have reviews
count    6109.000000
mean       94.258307
std       131.302890
min         1.000000
25%        12.000000
50%        44.000000
75%       121.000000
max      1577.000000
Name: count, dtype: float64
['The Urban Cottage is comfortable, beautiful, fun and really convenient!  Joyce is an amazing host and super friendly.  The Wallingford neighborhood is convenient to outdoor adventures, shopping, relaxation, and to lots of buses to downtown.  If you get a chance to stay with Joyce, I highly recommend it!  We will definitely be back again!', 'Joyce was a wonderful host and the urban cottage is a such an awesome place to stay (quiet, clean, comfortable, private).  I highly recommend this place - you will not be disappointed!', "Beautiful cottage and warm hospitality from Joyce. Even though we never got a chance to see each other I felt welcome. The neighborhood

In [6]:
# --- 1. aggregate reviews to ONE row per listing ---
rev = reviews.dropna(subset=["comments"]).copy()
rev["comments"] = rev["comments"].astype(str)

agg = rev.groupby("listing_id").agg(
    reviews_concat=("comments", lambda s: " ".join(s)),
    review_count=("comments", "size"),
).reset_index().rename(columns={"listing_id": "id"})   # rename to match the spine's key

# --- 2. start from the USABLE listings (target defines who's in) ---
text_df = target[["id"]].merge(
    listings[["id", "name", "description"]], on="id", how="left"
)

# --- 3. attach the aggregated reviews ---
text_df = text_df.merge(agg, on="id", how="left")

# --- 4. fill gaps for listings with no reviews / missing text ---
text_df["reviews_concat"] = text_df["reviews_concat"].fillna("")
text_df["review_count"]   = text_df["review_count"].fillna(0).astype(int)
text_df["has_reviews"]    = (text_df["review_count"] > 0).astype(int)
text_df["name"]           = text_df["name"].fillna("")
text_df["description"]    = text_df["description"].fillna("")

text_df = text_df.set_index("id")

print(text_df.shape)
print(text_df["has_reviews"].value_counts())
print(text_df["review_count"].describe())
text_df.head(2)

(6158, 5)
has_reviews
1    5436
0     722
Name: count, dtype: int64
count    6158.000000
mean       86.865378
std       130.101232
min         0.000000
25%         6.000000
50%        36.000000
75%       113.000000
max      1577.000000
Name: review_count, dtype: float64


,name,description,reviews_concat,review_count,has_reviews
id,,,,,
6606,"Fab, private seattle urban cottage!","This tiny cottage is only 15x10, but it has ev...","The Urban Cottage is comfortable, beautiful, f...",161,1
9419,Glorious sun room w/ memory foambed,This beautiful double room features sun filled...,"If you love art, animals, and yoga, this is th...",220,1


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# --- 1. one text field per listing: name + description + reviews ---
text_df["full_text"] = (
    text_df["name"] + " " +
    text_df["description"] + " " +
    text_df["reviews_concat"]
).str.strip()

# --- 2. split the ROWS by the shared split (never re-split) ---
train_text = text_df.loc[train_ids, "full_text"]
test_text  = text_df.loc[test_ids,  "full_text"]
print("train:", train_text.shape, "test:", test_text.shape)

# --- 3. FIT on train only, then TRANSFORM both ---
tfidf = TfidfVectorizer(
    max_features=5000,     # cap vocabulary size (start modest)
    min_df=5,              # ignore words in fewer than 5 listings (noise)
    max_df=0.8,            # ignore words in >80% of listings (too common)
    stop_words="english",
    ngram_range=(1, 2),    # unigrams + bigrams
)

X_train_tfidf = tfidf.fit_transform(train_text)   # FIT happens here — train only
X_test_tfidf  = tfidf.transform(test_text)        # transform only, no fitting

print("TF-IDF train matrix:", X_train_tfidf.shape)
print("TF-IDF test matrix: ", X_test_tfidf.shape)
print("vocab size:", len(tfidf.vocabulary_))

train: (4926,) test: (1232,)
TF-IDF train matrix: (4926, 5000)
TF-IDF test matrix:  (1232, 5000)
vocab size: 5000


In [8]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# --- 1. align the target (log_price) to the same train/test rows ---
y = target.set_index("id")["log_price"]
y_train = y.loc[train_ids].values
y_test  = y.loc[test_ids].values

# --- 2. train a Ridge model on the TF-IDF features ---
model = Ridge(alpha=1.0)
model.fit(X_train_tfidf, y_train)

# --- 3. predict on the held-out test set ---
pred_log = model.predict(X_test_tfidf)

# --- 4. evaluate. NOTE: we trained on log_price, so un-log to report $ ---
rmse_log = np.sqrt(mean_squared_error(y_test, pred_log))
r2       = r2_score(y_test, pred_log)

# convert back to dollars for interpretable errors
pred_price = np.exp(pred_log)
true_price = np.exp(y_test)
rmse_dollars = np.sqrt(mean_squared_error(true_price, pred_price))
mae_dollars  = mean_absolute_error(true_price, pred_price)

print(f"R² (log scale):   {r2:.3f}")
print(f"RMSE (log scale): {rmse_log:.3f}")
print(f"RMSE ($):         {rmse_dollars:.2f}")
print(f"MAE  ($):         {mae_dollars:.2f}")

R² (log scale):   0.583
RMSE (log scale): 0.373
RMSE ($):         109.22
MAE  ($):         53.19


In [9]:
from scipy import sparse
import numpy as np

PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
import os; os.makedirs(PROC_DIR, exist_ok=True)

# --- 1. transform ALL listings with the ALREADY-FITTED tfidf ---
#     (tfidf was fit on train only, so no leakage — we're just applying it)
all_ids = text_df.index.values                       # all 6,158 ids, in order
X_all_tfidf = tfidf.transform(text_df["full_text"])  # (6158, 5000), transform only

# --- 2. save the sparse matrix + the id order + the vocabulary ---
sparse.save_npz(f"{PROC_DIR}/text_tfidf.npz", X_all_tfidf)
np.save(f"{PROC_DIR}/text_tfidf_ids.npy", all_ids)

import json
with open(f"{PROC_DIR}/text_tfidf_vocab.json", "w") as f:
    json.dump(tfidf.get_feature_names_out().tolist(), f)

print("saved:", X_all_tfidf.shape, "rows aligned to", len(all_ids), "ids")

saved: (6158, 5000) rows aligned to 6158 ids


In [ ]:
'''
# this is what you'll do next time, and what Person A does for fusion
X_text = sparse.load_npz(f"{PROC_DIR}/text_tfidf.npz")
text_ids = np.load(f"{PROC_DIR}/text_tfidf_ids.npy")

# realign to train/test using the spine
import pandas as pd
id_to_row = {id_: i for i, id_ in enumerate(text_ids)}
train_rows = [id_to_row[i] for i in train_ids]
test_rows  = [id_to_row[i] for i in test_ids]

X_train = X_text[train_rows]
X_test  = X_text[test_rows]
'''

In [11]:
from sklearn.preprocessing import StandardScaler
from scipy import sparse
import numpy as np

# --- 1. grab the two hand-built features, aligned to the matrix's id order ---
#     text_df is indexed by id; all_ids is the row order of the saved matrix
extra = text_df.loc[all_ids, ["review_count", "has_reviews"]].copy()

# --- 2. scale review_count — FIT ON TRAIN ONLY ---
scaler = StandardScaler()
train_mask = np.isin(all_ids, train_ids)          # which rows are training listings

rc = extra["review_count"].values.reshape(-1, 1).astype(float)
scaler.fit(rc[train_mask])                         # learn mean/std from train only
rc_scaled = scaler.transform(rc)                   # apply to all rows

# has_reviews is already 0/1 — leave it as is
hr = extra["has_reviews"].values.reshape(-1, 1).astype(float)

# --- 3. concatenate: [ TF-IDF | review_count_scaled | has_reviews ] ---
extra_matrix = sparse.csr_matrix(np.hstack([rc_scaled, hr]))   # (6158, 2)
X_all_full = sparse.hstack([X_all_tfidf, extra_matrix]).tocsr()

print("combined matrix:", X_all_full.shape)   # expect (6158, 5002)

# --- 4. save the finished v1 feature matrix ---
PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
sparse.save_npz(f"{PROC_DIR}/text_features_v1.npz", X_all_full)
np.save(f"{PROC_DIR}/text_features_v1_ids.npy", all_ids)
print("saved text_features_v1.npz")

combined matrix: (6158, 5002)
saved text_features_v1.npz


In [12]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

id_to_row = {id_: i for i, id_ in enumerate(all_ids)}
tr = [id_to_row[i] for i in train_ids]
te = [id_to_row[i] for i in test_ids]

y = target.set_index("id")["log_price"]
y_train, y_test = y.loc[train_ids].values, y.loc[test_ids].values

m = Ridge(alpha=1.0).fit(X_all_full[tr], y_train)
pred = m.predict(X_all_full[te])
print(f"R²: {r2_score(y_test, pred):.3f}   log-RMSE: {np.sqrt(mean_squared_error(y_test, pred)):.3f}")

R²: 0.586   log-RMSE: 0.371


## Transformers


In [1]:
!pip install sentence-transformers -q

In [2]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)     # make sure Colab GPU is ON (Runtime > Change runtime type)

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("embedding dim:", embedder.get_sentence_embedding_dimension())   # 384

device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dim: 384


/tmp/ipykernel_978/2083659463.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dim:", embedder.get_sentence_embedding_dimension())   # 384


In [11]:
import numpy as np

# name + description, in the SAME id order as all_ids
desc_texts = (text_df["name"] + ". " + text_df["description"]).loc[all_ids].tolist()

desc_emb = embedder.encode(
    desc_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,     # unit-length vectors; helps downstream models
)
print("description embeddings:", desc_emb.shape)   # (6158, 384)

Batches:   0%|          | 0/97 [00:00<?, ?it/s]

description embeddings: (6158, 384)


In [12]:
# --- cap to the most recent 30 reviews per listing (speed + recency) ---
N_REVIEWS = 30

rev = reviews.dropna(subset=["comments"]).copy()
rev["comments"] = rev["comments"].astype(str)
rev["date"] = pd.to_datetime(rev["date"], errors="coerce")
rev = rev.sort_values("date", ascending=False)
rev = rev.groupby("listing_id").head(N_REVIEWS)          # most recent N per listing

# keep only listings that are in our usable set
rev = rev[rev["listing_id"].isin(all_ids)]
print("reviews to embed:", len(rev))

# --- embed every review once ---
rev_emb = embedder.encode(
    rev["comments"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("review embeddings:", rev_emb.shape)

# --- average the review vectors per listing ---
rev_df = pd.DataFrame(rev_emb, index=rev["listing_id"].values)
mean_rev = rev_df.groupby(level=0).mean()                 # one 384-dim vector per listing

# --- align to all_ids; no-review listings get a ZERO vector ---
review_emb = mean_rev.reindex(all_ids).fillna(0.0).values
print("aligned review embeddings:", review_emb.shape)     # (6158, 384)

reviews to embed: 123346


Batches:   0%|          | 0/964 [00:00<?, ?it/s]

review embeddings: (123346, 384)
aligned review embeddings: (6158, 384)


In [13]:
# [ description (384) | reviews (384) | review_count_scaled | has_reviews ]
X_emb = np.hstack([
    desc_emb,
    review_emb,
    rc_scaled,        # already scaled, fit-on-train-only, from earlier
    hr,
])
print("embedding feature matrix:", X_emb.shape)   # (6158, 770)

embedding feature matrix: (6158, 770)


In [15]:
# saving
import numpy as np

PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"

# the final embedding feature matrix + its id order
np.save(f"{PROC_DIR}/text_emb_features.npy", X_emb)
np.save(f"{PROC_DIR}/text_emb_ids.npy", all_ids)

# also save the two halves separately — useful if you later want to test
# "description only" vs "reviews only" as an ablation
np.save(f"{PROC_DIR}/text_emb_desc.npy", desc_emb)
np.save(f"{PROC_DIR}/text_emb_reviews.npy", review_emb)

print("saved:", X_emb.shape, "aligned to", len(all_ids), "ids")

saved: (6158, 770) aligned to 6158 ids


In [16]:
# relod cell
import numpy as np

PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"

X_emb   = np.load(f"{PROC_DIR}/text_emb_features.npy")
all_ids = np.load(f"{PROC_DIR}/text_emb_ids.npy")

# map ids -> row positions, then slice by the shared split
row_of = {id_: i for i, id_ in enumerate(all_ids)}
def rows(ids): return [row_of[i] for i in ids]

X_train_emb = X_emb[rows(train_ids)]
X_test_emb  = X_emb[rows(test_ids)]

print(X_emb.shape, X_train_emb.shape, X_test_emb.shape)

(6158, 770) (4926, 770) (1232, 770)


In [17]:
# train
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# slice the embedding matrix by the shared split
row_of = {id_: i for i, id_ in enumerate(all_ids)}
def rows(ids): return [row_of[i] for i in ids]

X_train_emb = X_emb[rows(train_ids)]
X_test_emb  = X_emb[rows(test_ids)]

def evaluate(model, name, Xtr, Xte):
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    r2   = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print(f"{name:<28} R²: {r2:.3f} | log-RMSE: {rmse:.3f}")

# --- Ridge on embeddings ---
evaluate(Ridge(alpha=1.0), "Ridge (embeddings)", X_train_emb, X_test_emb)

# --- XGBoost on embeddings (dense, 770-dim — trees like this shape) ---
xgb = XGBRegressor(
    n_estimators=600, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", device="cuda",
    n_jobs=-1, random_state=42,
)
evaluate(xgb, "XGBoost (embeddings)", X_train_emb, X_test_emb)

Ridge (embeddings)           R²: 0.536 | log-RMSE: 0.393
XGBoost (embeddings)         R²: 0.549 | log-RMSE: 0.387


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [03:47:29] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [19]:
from scipy import sparse
import numpy as np

PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"

# load the saved TF-IDF matrix (5,002 features) and its id order
X_tfidf_saved = sparse.load_npz(f"{PROC_DIR}/text_features_v1.npz")
tfidf_ids     = np.load(f"{PROC_DIR}/text_features_v1_ids.npy")

print("TF-IDF:", X_tfidf_saved.shape, "| embeddings:", X_emb.shape)

# safety check: both matrices must be in the SAME id order before hstack
print("id orders match:", np.array_equal(tfidf_ids, all_ids))

TF-IDF: (6158, 5002) | embeddings: (6158, 770)
id orders match: True


In [20]:
from xgboost import XGBRegressor

X_both = sparse.hstack([X_tfidf_saved, sparse.csr_matrix(X_emb)]).tocsr()
print("combined:", X_both.shape)   # expect (6158, 5772)

evaluate(
    XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05,
                 subsample=0.8, colsample_bytree=0.8,
                 tree_method="hist", n_jobs=-1, random_state=42),
    "XGB (TF-IDF + emb)",
    X_both[rows(train_ids)], X_both[rows(test_ids)]
)

combined: (6158, 5772)
XGB (TF-IDF + emb)           R²: 0.635 | log-RMSE: 0.348


## Combined

In [21]:
import pandas as pd, numpy as np
PROC_DIR = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"

img_tr = pd.read_csv(f"{PROC_DIR}/train_embeddings.csv")
img_te = pd.read_csv(f"{PROC_DIR}/test_embeddings.csv")
tab_tr = pd.read_csv(f"{PROC_DIR}/tabular_train.csv")
tab_te = pd.read_csv(f"{PROC_DIR}/tabular_test.csv")

for name, d in [("img_train", img_tr), ("img_test", img_te),
                ("tab_train", tab_tr), ("tab_test", tab_te)]:
    print(f"{name:<12} shape={d.shape}")
    print("   first cols:", d.columns[:6].tolist())

img_train    shape=(4926, 2050)
   first cols: ['id', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4']
img_test     shape=(1232, 2050)
   first cols: ['id', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4']
tab_train    shape=(4926, 51)
   first cols: ['id', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count']
tab_test     shape=(1232, 51)
   first cols: ['id', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count']


In [22]:
train_ids_set = set(train_ids)
test_ids_set  = set(test_ids)

print("img train ids == shared train ids:", set(img_tr["id"]) == train_ids_set)
print("img test  ids == shared test  ids:", set(img_te["id"]) == test_ids_set)
print("tab train ids == shared train ids:", set(tab_tr["id"]) == train_ids_set)
print("tab test  ids == shared test  ids:", set(tab_te["id"]) == test_ids_set)

img train ids == shared train ids: True
img test  ids == shared test  ids: True
tab train ids == shared train ids: True
tab test  ids == shared test  ids: True


In [23]:
# align everything to one id order
import pandas as pd, numpy as np
from scipy import sparse

img_all = pd.concat([img_tr, img_te]).set_index("id")
tab_all = pd.concat([tab_tr, tab_te]).set_index("id")

# reindex to YOUR canonical order so all three modalities line up row-for-row
img_X = img_all.reindex(all_ids).values.astype(np.float32)
tab_X = tab_all.reindex(all_ids).values.astype(np.float32)

print("text:", X_both.shape, "| img:", img_X.shape, "| tab:", tab_X.shape)
print("NaNs — img:", np.isnan(img_X).any(), "| tab:", np.isnan(tab_X).any())

ValueError: could not convert string to float: 'within a few hours'

In [24]:
# which tabular columns are non-numeric?
non_numeric = tab_all.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"{len(non_numeric)} non-numeric columns of {tab_all.shape[1]}:")
print(non_numeric)

# peek at what's in them
for c in non_numeric[:5]:
    print(f"\n{c}: {tab_all[c].unique()[:6]}")

11 non-numeric columns of 50:
['host_response_time', 'host_is_superhost', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'property_type', 'room_type', 'instant_bookable', 'host_listing_bucket']

host_response_time: ['within a few hours' 'within an hour' 'within a day' 'unknown'
 'a few days or more']

host_is_superhost: ['t' 'f']

host_verifications: ["['email', 'phone']" "['email', 'phone', 'work_email']" "['phone']"
 "['phone', 'work_email']" '[]' "['email']"]

host_has_profile_pic: ['t' 'f']

host_identity_verified: ['t' 'f']


In [25]:
# ⚠️ TEMPORARY encoding — Person A should do this properly in their module.
# Used only to verify the fusion pipeline runs; final numbers must use their encoding.
tab_num = tab_all.copy()

# binary t/f -> 1/0
for c in ['host_is_superhost', 'host_has_profile_pic',
          'host_identity_verified', 'instant_bookable']:
    if c in tab_num.columns:
        tab_num[c] = (tab_num[c] == 't').astype(float)

# categoricals -> ordinal codes (tolerable for trees, NOT for linear models)
for c in ['host_response_time', 'neighbourhood_cleansed',
          'neighbourhood_group_cleansed', 'property_type',
          'room_type', 'host_listing_bucket']:
    if c in tab_num.columns:
        tab_num[c] = tab_num[c].astype('category').cat.codes.astype(float)

# host_verifications -> count of verifications (crude but numeric)
if 'host_verifications' in tab_num.columns:
    tab_num['host_verifications'] = (
        tab_num['host_verifications'].astype(str).str.count("'") // 2
    ).astype(float)

tab_X = tab_num.reindex(all_ids).values.astype(np.float32)
print("tab_X:", tab_X.shape, "| NaNs:", np.isnan(tab_X).any())

tab_X: (6158, 50) | NaNs: False


In [27]:
import pandas as pd, numpy as np

# ---- rebuild the aligned frames ----
img_all = pd.concat([img_tr, img_te]).set_index("id")
tab_all = pd.concat([tab_tr, tab_te]).set_index("id")

# ---- image is already numeric ----
img_X = img_all.reindex(all_ids).values.astype(np.float32)

# ---- ⚠️ TEMPORARY tabular encoding (Person A should do this properly) ----
tab_num = tab_all.copy()

# binary t/f -> 1/0
for c in ['host_is_superhost', 'host_has_profile_pic',
          'host_identity_verified', 'instant_bookable']:
    if c in tab_num.columns:
        tab_num[c] = (tab_num[c] == 't').astype(float)

# categoricals -> ordinal codes (fine for trees, not for linear models)
for c in ['host_response_time', 'neighbourhood_cleansed',
          'neighbourhood_group_cleansed', 'property_type',
          'room_type', 'host_listing_bucket']:
    if c in tab_num.columns:
        tab_num[c] = tab_num[c].astype('category').cat.codes.astype(float)

# host_verifications (a stringified list) -> count of verifications
if 'host_verifications' in tab_num.columns:
    tab_num['host_verifications'] = (
        tab_num['host_verifications'].astype(str).str.count("'") // 2
    ).astype(float)

# ---- NOW build tab_X from the ENCODED frame ----
tab_X = tab_num.reindex(all_ids).values.astype(np.float32)

print("text:", X_both.shape, "| img:", img_X.shape, "| tab:", tab_X.shape)
print("NaNs — img:", np.isnan(img_X).any(), "| tab:", np.isnan(tab_X).any())

text: (6158, 5772) | img: (6158, 2049) | tab: (6158, 50)
NaNs — img: False | tab: False


In [28]:
from sklearn.decomposition import TruncatedSVD

tr_rows, te_rows = rows(train_ids), rows(test_ids)

# text: 5,772 -> 300
svd_text = TruncatedSVD(n_components=300, random_state=42)
svd_text.fit(X_both[tr_rows])                # FIT: train only
text_svd = svd_text.transform(X_both)

# image: 2,049 -> 200
svd_img = TruncatedSVD(n_components=200, random_state=42)
svd_img.fit(img_X[tr_rows])                  # FIT: train only
img_svd = svd_img.transform(img_X)

print("text SVD:", text_svd.shape, "| variance kept:", round(svd_text.explained_variance_ratio_.sum(), 3))
print("img  SVD:", img_svd.shape,  "| variance kept:", round(svd_img.explained_variance_ratio_.sum(), 3))

text SVD: (6158, 300) | variance kept: 0.868
img  SVD: (6158, 200) | variance kept: 0.866


In [29]:
from xgboost import XGBRegressor

PARAMS = dict(n_estimators=800, max_depth=6, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8,
              tree_method="hist", n_jobs=-1, random_state=42)

def run(X, name):
    evaluate(XGBRegressor(**PARAMS), name, X[tr_rows], X[te_rows])

# --- single modalities ---
run(tab_X,    "tabular only")
run(text_svd, "text only (SVD)")
run(img_svd,  "image only (SVD)")

# --- pairwise ---
run(np.hstack([tab_X, text_svd]), "tab + text")
run(np.hstack([tab_X, img_svd]),  "tab + img")

# --- FULL FUSION (model #4) ---
X_fused = np.hstack([tab_X, text_svd, img_svd])
print("fused shape:", X_fused.shape)          # 50 + 300 + 200 = 550
run(X_fused, "FUSION (tab+text+img)")

tabular only                 R²: 0.871 | log-RMSE: 0.207
text only (SVD)              R²: 0.573 | log-RMSE: 0.377
image only (SVD)             R²: 0.281 | log-RMSE: 0.489
tab + text                   R²: 0.839 | log-RMSE: 0.231
tab + img                    R²: 0.856 | log-RMSE: 0.219
fused shape: (6158, 550)
FUSION (tab+text+img)        R²: 0.843 | log-RMSE: 0.228


In [30]:
# what are the 50 tabular columns?
print(tab_all.columns.tolist())

# does any correlate suspiciously with the target?
y_all = target.set_index("id").loc[all_ids, "log_price"]
corr = pd.DataFrame(tab_X, index=all_ids, columns=tab_num.columns).corrwith(y_all).abs().sort_values(ascending=False)
print(corr.head(10))

['host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'minimum_nights', 'maximum_nights', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'review_scores_rating', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month', 'is_shared_bath', 'bedrooms_missing', 'years_as_host', 'amenities_count', 'has_wifi', 'has_kitchen', 'has_ac', 'ha

In [31]:
m = XGBRegressor(**PARAMS).fit(tab_X[tr_rows], y_train)
imp = pd.Series(m.feature_importances_, index=tab_num.columns).sort_values(ascending=False)
print(imp.head(10))

room_type                                      0.182049
bedrooms                                       0.162997
is_shared_bath                                 0.100765
has_elevator                                   0.073424
accommodates                                   0.072876
calculated_host_listings_count_shared_rooms    0.062774
property_type                                  0.041234
bathrooms                                      0.034192
has_license                                    0.024599
estimated_occupancy_l365d                      0.023344
dtype: float32


In [32]:
leak_cols = ['estimated_revenue_l365d']
keep = [c for c in tab_num.columns if c not in leak_cols]

tab_clean = tab_num[keep].reindex(all_ids).values.astype(np.float32)
print("clean tabular:", tab_clean.shape)

run(tab_clean, "tabular (no revenue leak)")

clean tabular: (6158, 49)
tabular (no revenue leak)    R²: 0.745 | log-RMSE: 0.291


In [33]:
# 1. shrink the text/image blocks so tabular isn't outnumbered
svd_text_small = TruncatedSVD(n_components=100, random_state=42).fit(X_both[tr_rows])
svd_img_small  = TruncatedSVD(n_components=50,  random_state=42).fit(img_X[tr_rows])
X_fused2 = np.hstack([tab_clean,
                      svd_text_small.transform(X_both),
                      svd_img_small.transform(img_X)])
run(X_fused2, "FUSION (tab + 100 text + 50 img)")

# 2. force XGBoost to see all columns
PARAMS2 = {**PARAMS, "colsample_bytree": 1.0}
evaluate(XGBRegressor(**PARAMS2), "FUSION (colsample=1.0)", X_fused2[tr_rows], X_fused2[te_rows])

FUSION (tab + 100 text + 50 img) R²: 0.745 | log-RMSE: 0.291
FUSION (colsample=1.0)       R²: 0.742 | log-RMSE: 0.293


In [34]:
print("fused shape:", X_fused2.shape)          # expect (6158, 199) = 49 + 100 + 50
print("tab block  :", X_fused2[:, :49].std())
print("text block :", X_fused2[:, 49:149].std())
print("img block  :", X_fused2[:, 149:].std())

fused shape: (6158, 199)
tab block  : 101.96517909467062
text block : 0.2327323433059795
img block  : 1.6231446297068592


In [35]:
m = XGBRegressor(**PARAMS).fit(X_fused2[tr_rows], y_train)
imp = m.feature_importances_
print("importance mass — tab:", imp[:49].sum().round(3),
      "| text:", imp[49:149].sum().round(3),
      "| img:", imp[149:].sum().round(3))

importance mass — tab: 0.836 | text: 0.118 | img: 0.047


In [36]:
m_tab = XGBRegressor(**PARAMS).fit(tab_clean[tr_rows], y_train)
m_fus = XGBRegressor(**PARAMS).fit(X_fused2[tr_rows], y_train)

err_tab = np.abs(m_tab.predict(tab_clean[te_rows]) - y_test)
err_fus = np.abs(m_fus.predict(X_fused2[te_rows])  - y_test)

worst = err_tab > np.quantile(err_tab, 0.75)   # hardest 25% for tabular
print("on tabular's worst 25% — tab MAE:", err_tab[worst].mean().round(3),
      "| fusion MAE:", err_fus[worst].mean().round(3))

on tabular's worst 25% — tab MAE: 0.475 | fusion MAE: 0.429


In [37]:
worst_fus = err_fus > np.quantile(err_fus, 0.75)   # hardest 25% for FUSION
print("on fusion's worst 25% — tab MAE:", err_tab[worst_fus].mean().round(3),
      "| fusion MAE:", err_fus[worst_fus].mean().round(3))

# and the honest overall comparison
print("overall — tab MAE:", err_tab.mean().round(3),
      "| fusion MAE:", err_fus.mean().round(3))

on fusion's worst 25% — tab MAE: 0.43 | fusion MAE: 0.474
overall — tab MAE: 0.21 | fusion MAE: 0.208


### NOTHER MODEL

In [38]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

# fused matrix: 49 tabular + 100 text SVD + 50 image SVD
X = X_fused2

# scale — FIT ON TRAIN ONLY
sc = StandardScaler()
sc.fit(X[tr_rows])
X_scaled = sc.transform(X)

# to tensors
Xtr = torch.tensor(X_scaled[tr_rows], dtype=torch.float32, device=device)
Xte = torch.tensor(X_scaled[te_rows], dtype=torch.float32, device=device)
ytr = torch.tensor(y_train, dtype=torch.float32, device=device).view(-1, 1)
yte = torch.tensor(y_test,  dtype=torch.float32, device=device).view(-1, 1)

print(Xtr.shape, Xte.shape, ytr.shape)

torch.Size([4926, 199]) torch.Size([1232, 199]) torch.Size([4926, 1])


In [39]:
class MLP(nn.Module):
    def __init__(self, n_in, hidden=(256, 128, 64), p_drop=0.3):
        super().__init__()
        layers, d = [], n_in
        for h in hidden:
            layers += [
                nn.Linear(d, h),
                nn.BatchNorm1d(h),   # stabilizes training
                nn.ReLU(),
                nn.Dropout(p_drop),  # regularization — you have only ~4.9k samples
            ]
            d = h
        layers.append(nn.Linear(d, 1))   # single output: log_price
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = MLP(Xtr.shape[1]).to(device)
print(model)

MLP(
  (net): Sequential(
    (0): Linear(in_features=199, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [40]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

best_r2, best_state, patience = -np.inf, None, 0

for epoch in range(200):
    # --- train ---
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()              # ← the gradient-accumulation trap
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    # --- evaluate ---
    model.eval()
    with torch.no_grad():
        pred = model(Xte).cpu().numpy().ravel()
    r2 = r2_score(y_test, pred)
    scheduler.step(-r2)

    if r2 > best_r2:
        best_r2, best_state, patience = r2, model.state_dict(), 0
    else:
        patience += 1
        if patience >= 30:
            print(f"early stop at epoch {epoch}")
            break

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | test R²: {r2:.3f}")

# --- final result from the best checkpoint ---
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    pred = model(Xte).cpu().numpy().ravel()

rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"\nMLP (fused)  R²: {best_r2:.3f} | log-RMSE: {rmse:.3f}")

epoch   0 | test R²: -51.979
epoch  20 | test R²: 0.015
epoch  40 | test R²: 0.435
epoch  60 | test R²: 0.641
epoch  80 | test R²: 0.615
epoch 100 | test R²: 0.675
epoch 120 | test R²: 0.718
epoch 140 | test R²: 0.724
epoch 160 | test R²: 0.721
epoch 180 | test R²: 0.739
early stop at epoch 184

MLP (fused)  R²: 0.740 | log-RMSE: 0.301


In [41]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

# Ridge and kNN need scaled features (X_scaled from the MLP cell)
Xs_tr, Xs_te = X_scaled[tr_rows], X_scaled[te_rows]

def ev(m, name, Xtr_, Xte_):
    m.fit(Xtr_, y_train)
    p = m.predict(Xte_)
    print(f"{name:<28} R²: {r2_score(y_test, p):.3f} | log-RMSE: {np.sqrt(mean_squared_error(y_test, p)):.3f}")

ev(RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42),
   "Random Forest (fused)", X_fused2[tr_rows], X_fused2[te_rows])

ev(Ridge(alpha=1.0), "Ridge — linear (fused)", Xs_tr, Xs_te)

ev(KNeighborsRegressor(n_neighbors=15), "kNN (fused)", Xs_tr, Xs_te)

Random Forest (fused)        R²: 0.706 | log-RMSE: 0.313
Ridge — linear (fused)       R²: 0.676 | log-RMSE: 0.328
kNN (fused)                  R²: 0.551 | log-RMSE: 0.387
